Code created by: Lorena Espinosa, Johana Rátiva, Eduards Chipatecua 

Class: Procesamiento del Lenguaje Natural

University: Universidad de los Andes

Date: August 23, 2026

In [2]:
import numpy as np
import pandas as pd
import math
import xml.etree.ElementTree as ET

# Metricas de Evaluacion

In [3]:
def tabla_pruebas(nombre_funcion, pruebas):
    resultados = []

    for prueba in pruebas:
        resultado = prueba["funcion"](*prueba["argumentos"])

        resultados.append({
            "Caso": prueba["caso"],
            "Entrada": prueba["entrada"],
            "Resultado": resultado
        })

    tabla = pd.DataFrame(resultados)

    display(
        tabla.style.set_caption(nombre_funcion)
    )

    return tabla

## Precision y Precision at K

In [4]:

def precision(re:list):
    """Calcula la precisión como la proporción de elementos relevantes
    recuperados sobre el total de elementos recuperados.
    Esto sobre una lista de relevancia binaria (1 para relevante, 0 para no relevante).
    Tiene en cuenta que la lista puede estar vacía,
    en cuyo caso devuelve 0.0. Evitando asi la división por cero.
    --
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).

    Returns:
    float
        La precisión calculada.
    """
    result = 0.0
    size = len(re)
    if size > 0:
        relevance =  np.array(re)
        result = (np.sum(relevance == 1))/size
    return result

def precision_at_k(re:list, k:int):
    """Calcula la precisión en los primeros k elementos de la lista de relevancia binaria.
    
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).
    k: int
        Número de elementos a considerar para el cálculo de la precisión.
    
    Returns:
    float
        La precisión calculada en los primeros k elementos.
        Es la fracción de elementos devueltos relevantes entre los k primeros.   
        Esta metrica aumenta cuando los elementos relevantes se encuentran en las primeras posiciones. Y disminuye cuando los elementos relevantes se encuentran en las últimas posiciones.
    """
    result = 0.0
    size = len(re)
    if size > 0 and k > 0 and size >= k:
        relevance = np.array(re)
        result = np.sum(relevance[:k] ==1)/k
    return result        

### Examples

In [5]:
pruebas_precision = [
    {
        "caso": "Ejemplo",
        "entrada": [0, 1, 0, 0, 1],
        "argumentos": [[0, 1, 0, 0, 1]],
        "funcion": precision
    },
    {
        "caso": "Prueba 1",
        "entrada": [1, 1, 0, 1, 0],
        "argumentos": [[1, 1, 0, 1, 0]],
        "funcion": precision
    },
    {
        "caso": "Prueba 2",
        "entrada": [1, 1, 1, 1],
        "argumentos": [[1, 1, 1, 1]],
        "funcion": precision
    },
    {
        "caso": "Vector vacío",
        "entrada": [],
        "argumentos": [[]],
        "funcion": precision
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": [0, 0, 0, 0],
        "argumentos": [[0, 0, 0, 0]],
        "funcion": precision
    }
]

tabla_precision = tabla_pruebas(
    "Precision",
    pruebas_precision
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1]",0.400000
1,Prueba 1,"[1, 1, 0, 1, 0]",0.600000
2,Prueba 2,"[1, 1, 1, 1]",1.000000
3,Vector vacío,[],0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0]",0.000000


In [6]:
pruebas_precision_k = [
    {
        "caso": "Ejemplo",
        "entrada": "[0, 1, 0, 0, 1], k=3",
        "argumentos": [[0, 1, 0, 0, 1], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "Prueba 1",
        "entrada": "[1, 1, 0, 1, 0], k=3",
        "argumentos": [[1, 1, 0, 1, 0], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "Prueba 2",
        "entrada": "[0, 1, 1, 0, 1], k=4",
        "argumentos": [[0, 1, 1, 0, 1], 4],
        "funcion": precision_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=3",
        "argumentos": [[], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[1, 0, 1], k=10",
        "argumentos": [[1, 0, 1], 10],
        "funcion": precision_at_k
    }
]

tabla_precision_k = tabla_pruebas(
    "Precision@K",
    pruebas_precision_k
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1], k=3",0.333333
1,Prueba 1,"[1, 1, 0, 1, 0], k=3",0.666667
2,Prueba 2,"[0, 1, 1, 0, 1], k=4",0.500000
3,Vector vacío,"[], k=3",0.000000
4,K mayor que longitud,"[1, 0, 1], k=10",0.000000


## Recall at K

In [7]:
def recall_at_k(relevance_query:list, number_relevant_docs:int, k:int):
    """Calcula el recall en los primeros k elementos de la lista de relevancia binaria.

    Parameters:
    relevance_query: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).
    number_relevant_docs: int
        Número total de documentos relevantes. Este valor R(q) está dado por un conjunto de referencia de relevancia para la consulta q.
    k: int
        Número de elementos a considerar para el cálculo del recall.

    Returns:
    float
        El recall calculado en los primeros k elementos.
        Es la fracción de elementos relevantes recuperados en k sobre el total de elementos relevantes R(q).
        Esta metrica puede subir o mantenerse constante. No puede disminuir.
    """
    result = 0.0
    size = len(relevance_query)
    if size > 0 and number_relevant_docs > 0 and k > 0 and size >= k:
        relevance = np.array(relevance_query)
        result = np.sum(relevance[:k] == 1)/number_relevant_docs
    return result

### Examples

In [8]:
pruebas_recall_k = [
    {
        "caso": "Ejemplo",
        "entrada": "[0, 1, 0, 0, 1], number_relevant_docs=4, k=3",
        "argumentos": [[0, 1, 0, 0, 1], 4, 3],
        "funcion": recall_at_k
    },
    {
        "caso": "Prueba 1",
        "entrada": "[1, 0, 1, 1, 0], number_relevant_docs=4, k=5",
        "argumentos": [[1, 0, 1, 1, 0], 4, 5],
        "funcion": recall_at_k
    },
    {
        "caso": "Prueba 2",
        "entrada": "[1, 0, 0, 1, 0], number_relevant_docs=5, k=5",
        "argumentos": [[1, 0, 0, 1, 0], 5, 5],
        "funcion": recall_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], number_relevant_docs=0, k=3",
        "argumentos": [[], 0, 3],
        "funcion": recall_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[1, 0, 1], number_relevant_docs=2, k=10",
        "argumentos": [[1, 0, 1], 2, 10],
        "funcion": recall_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], number_relevant_docs=0, k=4",
        "argumentos": [[0, 0, 0, 0], 0, 4],
        "funcion": recall_at_k
    }
]

tabla_recall_k = tabla_pruebas(
    "Recall@K",
    pruebas_recall_k
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1], number_relevant_docs=4, k=3",0.250000
1,Prueba 1,"[1, 0, 1, 1, 0], number_relevant_docs=4, k=5",0.750000
2,Prueba 2,"[1, 0, 0, 1, 0], number_relevant_docs=5, k=5",0.400000
3,Vector vacío,"[], number_relevant_docs=0, k=3",0.000000
4,K mayor que longitud,"[1, 0, 1], number_relevant_docs=2, k=10",0.000000
5,Sin documentos relevantes,"[0, 0, 0, 0], number_relevant_docs=0, k=4",0.000000


## Average precision

In [9]:
def average_precision(re: list):
    """Calcula la precisión promedio (Average Precision) para una lista de relevancia binaria.
    La precisión promedio es la media de las precisiones calculadas en cada posición k donde hay un elemento relevante.
    --
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).

    Returns:
    float
        La precisión promedio calculada.
    """
    result = 0.0
    relevance = np.array(re)
    precisions=[]
    relevant_count = 0
    if len(relevance) > 0 and np.sum(relevance ==1) > 0 : # aqui evalua que haya al menos un elemento relevante en la lista de relevancia, asi, suma uno a la cuenta de elementos relevantes y calcula la precision en cada posicion k donde hay un elemento relevante.
        for k, value in enumerate(relevance):
            if  value==1:
                relevant_count += 1
                precisions.append(relevant_count/(k+1))
        result = sum(precisions)/len(precisions)
    return result

### Examples

In [10]:
pruebas_average_precision = [
    {
        "caso": "Ejemplo",
        "entrada": "[1, 0, 1, 1, 0]",
        "argumentos": [[1, 0, 1, 1, 0]],
        "funcion": average_precision
    },
    {
        "caso": "Prueba 1",
        "entrada": "[0, 1, 0, 1, 1]",
        "argumentos": [[0, 1, 0, 1, 1]],
        "funcion": average_precision
    },
    {
        "caso": "Prueba 2",
        "entrada": "[1, 1, 0, 0, 1]",
        "argumentos": [[1, 1, 0, 0, 1]],
        "funcion": average_precision
    },
    {
        "caso": "Vector vacío",
        "entrada": "[]",
        "argumentos": [[]],
        "funcion": average_precision
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0, 0]",
        "argumentos": [[0, 0, 0, 0, 0]],
        "funcion": average_precision
    }
]

tabla_average_precision = tabla_pruebas(
    "Average Precision",
    pruebas_average_precision
)

,Caso,Entrada,Resultado
0,Ejemplo,"[1, 0, 1, 1, 0]",0.805556
1,Prueba 1,"[0, 1, 0, 1, 1]",0.533333
2,Prueba 2,"[1, 1, 0, 0, 1]",0.866667
3,Vector vacío,[],0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0, 0]",0.000000


## Mean average precision (MAP)

In [11]:
def mean_average_precision(relevance_queries:list[list]):
    """"Calcula la precisión promedio media (Mean Average Precision) para un conjunto de consultas.
    La precisión promedio media es la media de las precisiones promedio calculadas para cada consulta.
    
    Parameters:
    relevance_queries: list[list]
        Lista de listas de relevancia binaria (1 para relevante, 0 para no relevante) para cada consulta.
    
    Returns:
    float
        La precisión promedio media calculada.
        """
    precisions=[]
    result=0.0
    if len(relevance_queries) > 0:
        precisions = [average_precision(query) for query in relevance_queries]
        result = sum(precisions)/len(precisions)
    return result


### Examples

In [12]:
pruebas_map = [
    {
        "caso": "Ejemplo",
        "entrada": "[[1, 0, 1, 1, 0], [1, 1, 0, 0, 1]]",
        "argumentos": [
            [
                [1, 0, 1, 1, 0],
                [1, 1, 0, 0, 1]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Prueba 1",
        "entrada": "[[1, 1, 0, 1, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1]]",
        "argumentos": [
            [
                [1, 1, 0, 1, 0],
                [0, 1, 1, 0, 0],
                [1, 0, 0, 1, 1]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Prueba 2",
        "entrada": "[[1, 0, 0, 1], [1, 1, 0, 0], [0, 1, 1, 0]]",
        "argumentos": [
            [
                [1, 0, 0, 1],
                [1, 1, 0, 0],
                [0, 1, 1, 0]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Lista de consultas vacía",
        "entrada": "[]",
        "argumentos": [[]],
        "funcion": mean_average_precision
    },
    {
        "caso": "Consulta sin documentos relevantes",
        "entrada": "[[1, 0, 1, 0], [0, 0, 0, 0]]",
        "argumentos": [
            [
                [1, 0, 1, 0],
                [0, 0, 0, 0]
            ]
        ],
        "funcion": mean_average_precision
    }
]

tabla_map = tabla_pruebas(
    "Mean Average Precision (MAP)",
    pruebas_map
)

,Caso,Entrada,Resultado
0,Ejemplo,"[[1, 0, 1, 1, 0], [1, 1, 0, 0, 1]]",0.836111
1,Prueba 1,"[[1, 1, 0, 1, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1]]",0.733333
2,Prueba 2,"[[1, 0, 0, 1], [1, 1, 0, 0], [0, 1, 1, 0]]",0.777778
3,Lista de consultas vacía,[],0.000000
4,Consulta sin documentos relevantes,"[[1, 0, 1, 0], [0, 0, 0, 0]]",0.416667


## DCG at K

In [13]:
def dcg_at_k(relevance_query: list, k: int, gain: str):
    """Calcula el la ganancia acumulada descontada (DCG) en los primeros k elementos de la lista de relevancia.
    Esta metrica tiene en cuenta la relevancia de los elementos y su posición en la lista.
    Los elementos relevantes en posiciones más altas contribuyen más a la ganancia acumulada.
    Las posiciones bajas castigan la ganancia acumulada.
    
    Parameters:
    relevance_query: list
        Lista de relevancia en escala particular no binaria.
    k: int
        Número de elementos a considerar.
    gain: str
        Tipo de ganancia a utilizar ('linear' o 'exponential').
        linear: La ganancia es proporcional a la relevancia del elemento.
        exponential: La ganancia es exponencial a la relevancia del elemento. 

    returns:
    float
        La ganancia acumulada descontada calculada en los primeros k elementos.    
    """
    relevance = np.array(relevance_query)
    size = len(relevance)
    result = 0.0
    if size > 0 and k > 0 and size >= k:
        for i, rel in enumerate(relevance[:k]):
            if gain == 'linear':
                result += rel / math.log2(i + 2)
            elif gain == 'exponential':
                result += (2 ** rel - 1) / math.log2(i + 2)
    return result


### Example

In [14]:
pruebas_dcg_k = [
    {
        "caso": "Ejemplo 1 - ganancia lineal",
        "entrada": "[3, 2, 0, 1, 2], k=5, gain='linear'",
        "argumentos": [[3, 2, 0, 1, 2], 5, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Ejemplo 2 - ganancia exponencial",
        "entrada": "[3, 0, 2, 1, 3], k=5, gain='exponential'",
        "argumentos": [[3, 0, 2, 1, 3], 5, "exponential"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=5, gain='linear'",
        "argumentos": [[], 5, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[3, 2, 1], k=10, gain='linear'",
        "argumentos": [[3, 2, 1], 10, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], k=4, gain='linear'",
        "argumentos": [[0, 0, 0, 0], 4, "linear"],
        "funcion": dcg_at_k
    }
]

tabla_dcg_k = tabla_pruebas(
    "DCG@K",
    pruebas_dcg_k
)

,Caso,Entrada,Resultado
0,Ejemplo 1 - ganancia lineal,"[3, 2, 0, 1, 2], k=5, gain='linear'",5.466242
1,Ejemplo 2 - ganancia exponencial,"[3, 0, 2, 1, 3], k=5, gain='exponential'",11.638646
2,Vector vacío,"[], k=5, gain='linear'",0.000000
3,K mayor que longitud,"[3, 2, 1], k=10, gain='linear'",0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0], k=4, gain='linear'",0.000000


## NDCG at K

In [15]:
def ndcg_at_k(relevance_query, k, gain='exponential'):
    """Calcula la ganancia acumulada descontada normalizada (NDCG) en los primeros k elementos de la lista de relevancia.
    Esta metrica usa como referencia la ganancia acumulada descontada ideal (IDCG) que se calcula ordenando la lista de relevancia de mayor a menor.
    La NDCG se calcula como la relación entre la DCG y la IDCG.
    
    Parameters:
    relevance_query: list
        Lista de relevancia en escala particular no binaria.
    k: int
        Número de elementos a considerar.
    gain: str
        Tipo de ganancia a utilizar ('linear' o 'exponential').
        linear: La ganancia es proporcional a la relevancia del elemento.
        exponential: La ganancia es exponencial a la relevancia del elemento.   
        
        returns:
    float
        La ganancia acumulada descontada normalizada calculada en los primeros k elementos.
    """
    result = 0.0
    if len(relevance_query)>0 : 
        ideal_query = sorted(relevance_query, reverse=True)
        dcg_Ideal = dcg_at_k(ideal_query,k,gain)
        if dcg_Ideal > 0 :
            result = (dcg_at_k(relevance_query,k,gain))/(dcg_Ideal)
    return result    

### Example

In [16]:
pruebas_ndcg_k = [
    {
        "caso": "Ejemplo 1",
        "entrada": "[3, 2, 0, 1, 2], k=5, gain='linear'",
        "argumentos": [[3, 2, 0, 1, 2], 5],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Ejemplo 2",
        "entrada": "[0, 1, 3, 2, 0], k=5, gain='linear'",
        "argumentos": [[0, 1, 3, 2, 0], 5],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=5, gain='linear'",
        "argumentos": [[], 5, "linear"],
        "funcion": ndcg_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[3, 2, 1], k=10, gain='linear'",
        "argumentos": [[3, 2, 1], 10, "linear"],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], k=4, gain='linear'",
        "argumentos": [[0, 0, 0, 0], 4, "linear"],
        "funcion": ndcg_at_k
    }
]

tabla_ndcg_k = tabla_pruebas(
    "NDCG@K",
    pruebas_ndcg_k
)

,Caso,Entrada,Resultado
0,Ejemplo 1,"[3, 2, 0, 1, 2], k=5, gain='linear'",0.968638
1,Ejemplo 2,"[0, 1, 3, 2, 0], k=5, gain='linear'",0.577353
2,Vector vacío,"[], k=5, gain='linear'",0.000000
3,K mayor que longitud,"[3, 2, 1], k=10, gain='linear'",0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0], k=4, gain='linear'",0.000000
